In [48]:
!pip install -q --upgrade youtube-transcript-api


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3.12 install --upgrade pip


In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi

In [ ]:
# Constants
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# check OpenAI API key format
if OPENAI_API_KEY and OPENAI_API_KEY.startswith('sk-proj-') and len(OPENAI_API_KEY)>10:
    print("API key looks good ✅")
else:
    print("There might be a problem with your API key?")
    
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL_GPT = 'gpt-4o-mini'

parent_dir = os.path.dirname(os.getcwd())  # Get the parent directory (GenAI-Discovery)
env_path = os.path.join(parent_dir, '.env')

# Load environment variables
load_dotenv(dotenv_path=env_path, override=True)

API key looks good so far


In [ ]:
# Sessions are available on YouTube
URL = 'https://www.youtube.com/watch?v=O7AysEdsMqQ'
VIDEO_ID = URL.split('v=')[1][:11]

yt_obj = YouTubeTranscriptApi()
transcripts = yt_obj.list(video_id=VIDEO_ID)

# get available language
lang_code = list(transcripts)[0].language_code

# Fetch fist language track (hu)
transcript_obj = transcripts.find_transcript([lang_code])
transcript_data = transcript_obj.fetch()

# Combine all snippet text into a single continuous string
full_transcript_text = " ".join([entry.text for entry in transcript_data])

In [49]:
# Display the first 1000 characters of the transcript
display(Markdown(full_transcript_text[:1000] + " ..."))

Tisztelettel köszöntöm az ország országgyűlést, valamennyi képviselőtársamat és minden kedves érdeklődőt, aki figyelemmel kíséri a munkánkat. Az Országgyűlés hetedik rendkívüli ülésének második ülésnapját megnyitom. Tájékoztatom önöket, hogy az ülés vezetésében Néher András és Varga Gábor jegyzők lesznek segítségemre. Tisztelt Országgyűlés! A mai napon a kormány nevében felszólalásra jelentkezett Dr. Hegedű Zsolt Csaba egészségügyi miniszter úr. Az egészségügy helyzetéről és megújításáról címmel. miniszter úr, öné a szó 10 perces időkeretben. [harang] Tisztelt elnökasszony, tisztelt ház, tisztelt képviselőtársaim! A Magyar Egészségről ma csak őszintén érdemes beszélni. Egy olyan rendszert örököltünk, amelyben túl sok beteg vár túl sokáig. Betegek bolyonganak az ellátás különböző szintjei között. Orvosok, ápolók és szakdolgozók tzrei hiányoznak. A kórházakban pedig évtizedes műszaki, pénzügyi, szervezeti és bizalmi adósság halmozódott fel. Hiányoznak a kiszámítható betegutak. Sok helyen ...

In [54]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Hungarian Pairlament meeting (in Hungarian Language).
Please write minutes in markdown without code blocks, including:
- a summary with attendees
- discussion points
- takeaways
- action items with owners
Your answers should be in English, regardless of the transcript language.

Transcription:
{full_transcript_text}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [55]:
# Get gpt-4o-mini to answer, with streaming
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    max_tokens=2000,
    stream=True,
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

## Meeting Minutes

### Summary
**Date:** Not specified  
**Location:** Hungarian Parliament  
**Attendees:**  
- Dr. Hegedű Zsolt Csaba (Minister of Health)  
- Néher András (Secretary)  
- Varga Gábor (Secretary)  
- Apáti István (Opposition Representative)  
- Rétvári Bence (KDNP Representative)  
- Other members of the Parliament  

### Key Discussion Points
1. **Health System Reform:**
   - Dr. Hegedű Zsolt discussed the dire state of the Hungarian health system, including infrastructure issues and staff shortages.
   - Emphasized the government's commitment to transparency, professionalism, and patient-centered care.  
   - Announced a budget of at least 500 billion Ft for the state healthcare system annually.

2. **Government's Policy Transition:**
   - Dr. Hegedű articulated a shift towards a new health policy focused on accountability and listening to healthcare workers.
   - Initiatives like reducing VAT on prescription medications were highlighted.

3. **Critiques from Opposition:**
   - Opposition members, including Apáti István, expressed skepticism regarding the Minister's assurances, emphasizing previous failures in health governance.
   - Concerns were raised about the actual impact of budget increases and promised reforms.

4. **Historical Context of Administrative Terminology:**
   - A heated discussion arose regarding the return to historical administrative terms such as "Vármegye" (County) and "Főispán" (Lord Lieutenant) vs "Megye" (County).
   - Rétvári Bence and others drew parallels between the current government's actions and historical instances of power concentration during communist regimes.

5. **Civic Engagement and Public Sentiment:**
   - Multiple representatives cited the need for societal consultations before implementing changes.
   - Representations of historical legacy and national identity were heavily debated, highlighting contrasting views on governance and historical narratives.

6. **Administrative Efficiency:**
   - Questions about the efficiency of current governmental structures and the implications of renaming administrative titles were raised.
   - Various representatives argued for the relevance and implications of these terminological changes on governance and public service efficacy.

### Takeaways
- The Hungarian healthcare system is undergoing significant scrutiny and reform efforts, with a strong call for transparency and accountability.
- The shift in administrative terminology reflects deeper historical and cultural debates within Hungarian society.
- There remains a divide between the government and opposition regarding the effectiveness and urgency of proposed reforms.

### Action Items
1. **Healthcare Improvements:**
   - **Owner:** Dr. Hegedű Zsolt Csaba
   - **Action:** Implement proposed health system reforms, ensure funding allocation of 500 billion Ft per year.

2. **Public Consultation:**
   - **Owner:** Government Ministries
   - **Action:** Facilitate public consultations regarding health reforms and administrative changes.

3. **Clarification on Administrative Changes:**
   - **Owner:** Lőrinc Viktória (Minister)
   - **Action:** Provide clear communication on the objectives and benefits of reverting to historical administrative terms.

4. **Monitoring and Accountability:**
   - **Owner:** Parliament Representatives
   - **Action:** Hold sessions to monitor the implementation of health policies and the management of public funds.

Notable remarks from the meeting highlighted the intricate relationship between historical perceptions, governance, and public policy in Hungary, contributing to ongoing debates in the Parliament.